In [ ]:
"""
A path to json file containing list of objects of llm response needs to be passed with data in the format
question: < question asked >
answer: < answer by RAG >
context: < source reference as a string >

Eg: [{
    'question': 'who is joe biden',
    'answer': 'joe biden was a president of united states',
    'context': 'joe biden is from USA \n he was a president'
}]
"""

response_data_file = 'llm_res_cc.json'
app_name = "chatApp"
app_version = "enhanced"

In [ ]:
import numpy as np
import json
import pandas as pd
from trulens.core import TruSession
from trulens.core import Feedback
from trulens.providers.openai import OpenAI
from trulens_eval.tru_virtual import VirtualRecord
from trulens.apps.virtual import VirtualApp
from trulens.core import Select

session = TruSession()
session.reset_database()

# Initialize provider class
provider = OpenAI()

virtual_app = {
    'llm': {'modelname': 'some llm component model name'},
    'template': 'information about the template used in the app',
    'debug': 'optional fields for additional debugging information'
}

virtual_app = VirtualApp(virtual_app)
virtual_app[Select.RecordCalls.llm.maxtokens] = 1024
retriever_component = Select.RecordCalls.retriever
virtual_app[retriever_component] = 'this is the retriever component'
context_call = retriever_component.get_context

In [ ]:
with open(response_data_file) as f: llm_res_data = json.loads(f.read())

df = pd.DataFrame(llm_res_data)
responses = df.to_dict('records')

data = []

for record in responses:
    rec = VirtualRecord(
        main_input=record['question'],
        main_output=record['answer'],
        calls={
                context_call: {
                    "args": record['question'],
                    "rets": [record['context']]
                }
            }
        )
    data.append(rec)

context = context_call.rets[:]

f_context_relevance = (
    Feedback(provider.context_relevance_with_cot_reasons, name="Context Relevance")
    .on_input()
    .on(context)
    .aggregate(np.mean)
)

f_answer_relevance = Feedback(
    provider.relevance_with_cot_reasons, name="Answer Relevance"
).on_input_output()

f_groundedness = (
    Feedback(
        provider.groundedness_measure_with_cot_reasons, name="Groundedness"
    )
    .on(context.collect())  # collect context chunks into a list
    .on_output()
)

from trulens_eval.tru_virtual import TruVirtual

virtual_recorder = TruVirtual(
    app_name=app_name,
    app_version=app_version,
    app_id=app_version,
    app=virtual_app,
    feedbacks=[f_groundedness, f_context_relevance, f_answer_relevance]
)

import time
for rec in data:
    virtual_recorder.add_record(rec)
    time.sleep(1)

/Users/hitheshbhat/Documents/engineering/chat-knowledge-base/venv/lib/python3.10/site-packages/trulens/feedback/llm_provider.py:1521: UserWarning: Failed to process and remove trivial statements. Proceeding with all statements.
  warnings.warn(


In [ ]:
from trulens.dashboard import run_dashboard

run_dashboard(session=session, force=True)